## Gold Cascade Modeling (Deliverable 1.3.3)

This notebook implements the full three-stage cascade anomaly detection pipeline. The objective is to determine whether a layered approach improves alert quality when compared to the single-model baseline.

**Purpose:**  
To operationalize the cascade architecture defined in Section C, generating alert outputs for each stage of the cascade: broad detection (Stage 1), refined detection (Stage 2), and rule/profile/historical confirmation (Stage 3).

**Stages Implemented:**

1. **Stage 1 — Broad Isolation Forest**  
   High-sensitivity anomaly screen using all 50 Gold numeric features.

2. **Stage 2 — Narrow Isolation Forest**  
   Secondary detector trained on the reduced feature subset identified during Silver EDA.

3. **Stage 3 — Rule / Profile / Historical Confirmation**  
   Final confirmation based on behavior profiles, persistence checks, drift features, and cross-sensor consistency.

**Key Goals:**

- Load the Gold preprocessed dataset and Gold feature artifacts.
- Train Stage 1 and Stage 2 Isolation Forest models.
- Apply Stage 3 rule/profile confirmation logic based on Silver EDA outputs.
- Generate and store alert outputs for all three cascade stages.
- Export all cascade artifacts for comparative evaluation.

**Relevance to Section C:**  
This notebook produces the layered alert outputs required for evaluating the cascade’s effect on false positives, noisy alerts, and anomaly sensitivity. These outputs are necessary for the statistical tests, alert-volume comparisons, and visual communication described in Section C.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union

#import os
#import glob

from pathlib import Path
import yaml
import re

import logging
import wandb

import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 
import seaborn as sns 

import joblib 

from sklearn.model_selection import train_test_split, KFold

from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, RobustScaler

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, roc_auc_score, average_precision_score

from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor

import pyarrow.parquet as pq
import pyarrow as pa


import hashlib


# Custom Utilities Module
from utils.paths import get_paths
from utils.file_io import load_data, save_data, save_json, load_json
from utils.eda_logging import profile_dataframe
from utils.logging_setup import configure_logging
from utils.wandb_utils import finalize_wandb_stage

# Ledger 
from utils.ledger import Ledger

# Show more columns
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)


In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
def log_gold_paths(paths, logger: logging.Logger) -> None:
    logger.info("Project Root Path Loaded: %s", paths.root)
    logger.info("Project Logging Path Loaded: %s", paths.logs)
    logger.info("Project Artifacts Path Loaded: %s", paths.artifacts)
    logger.info("Project Notebooks Path Loaded %s", paths.notebooks)
    logger.info("Project Data Path Loaded: %s", paths.data)
    logger.info("Data Bronze Path Loaded: %s", paths.data_bronze)
    logger.info("Data Bronze Training Path Loaded: %s", paths.data_bronze_train)
    logger.info("Data Bronze Testing Path Loaded: %s", paths.data_bronze_test)
    logger.info("Data Silver Path Loaded: %s", paths.data_silver)
    logger.info("Data Silver Training Path Loaded: %s", paths.data_silver_train)
    logger.info("Data Silver Testing Path Loaded: %s", paths.data_silver_test)
    logger.info("Data Gold Path Loaded: %s", paths.data_gold)

In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
# Configurables

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Stage Details
STAGE = "gold"
LAYER_NAME = "gold"
GOLD_VERSION = "gold__001"
RECIPE_ID = "gold_modeling__v001_cascade"


DATASET_NAME_CONFIG = "pump"
DATASET_NAME = str(DATASET_NAME_CONFIG).strip().lower()

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Weights and Biases
WANDB_PROJECT = "capstone"
WANDB_ENTITY = "dcoo230-western-governors-university"
WANDB_RUN_NAME = f"{GOLD_VERSION}"


#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# File names
GOLD_FILE_NAME = f"{DATASET_NAME}__gold__preprocessed.parquet"

STAGE1_FEATURES_FILE_NAME = f"{DATASET_NAME}__gold__stage1_features.json"
STAGE2_FEATURES_FILE_NAME = f"{DATASET_NAME}__gold__stage2_features.json"
STAGE3_PRIMARY_FILE_NAME = f"{DATASET_NAME}__gold__stage3_primary_rule_sensors.json"
STAGE3_SECONDARY_FILE_NAME = f"{DATASET_NAME}__gold__stage3_secondary_rule_sensors.json"

CASCADE_RESULTS_FILE_NAME = f"{DATASET_NAME}__gold__cascade_results.csv"

COMPARISON_FILE_NAME = f"{DATASET_NAME}__gold__comparison__baseline_vs_cascade.csv"


#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Split configuration
TRAIN_FRACTION = 0.70
RANDOM_SEED = 42

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Scaling posture
USE_ROBUST_SCALER = False

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Cascade thresholding
STAGE1_THRESHOLD_PERCENTILE = 95.0   # broader / more sensitive
STAGE2_THRESHOLD_PERCENTILE = 98.5   # narrower / stricter

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Isolation Forest size
STAGE1_ESTIMATOR_COUNT = 200
STAGE2_ESTIMATOR_COUNT = 300

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 







In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
# Paths Setup

# Get Path's Object
paths = get_paths()

# Gold
GOLD_DATA_PATH = paths.data_gold / GOLD_FILE_NAME
GOLD_ARTIFACTS_PATH = paths.artifacts / "gold" / DATASET_NAME

# Stage artifacts
STAGE1_FEATURES_PATH = GOLD_ARTIFACTS_PATH / STAGE1_FEATURES_FILE_NAME
STAGE2_FEATURES_PATH = GOLD_ARTIFACTS_PATH / STAGE2_FEATURES_FILE_NAME
STAGE3_PRIMARY_PATH = GOLD_ARTIFACTS_PATH / STAGE3_PRIMARY_FILE_NAME
STAGE3_SECONDARY_PATH = GOLD_ARTIFACTS_PATH / STAGE3_SECONDARY_FILE_NAME

# Outputs
CASCADE_RESULTS_PATH = GOLD_ARTIFACTS_PATH / CASCADE_RESULTS_FILE_NAME

CASCADE_RESULTS_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__cascade_results.csv"

STAGE1_MODEL_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__stage1_isolation_forest.joblib"
STAGE2_MODEL_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__stage2_isolation_forest.joblib"

CASCADE_THRESHOLDS_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__cascade_thresholds.json"
CASCADE_SUMMARY_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__cascade_summary.json"
CASCADE_METADATA_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__cascade_metadata.json"

CASCADE_REFERENCE_PROFILE_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__cascade_reference_profile.csv"

# Logs
LOGS_PATH = paths.logs

# Path Failsafes
GOLD_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)



In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
# Logging Setup

# Create gold log path 
gold_log_path = paths.logs / "gold_modeling_cascade.log"

# Initial Logger
configure_logging(
    "capstone",
    gold_log_path,
    level=logging.DEBUG,
    overwrite_handlers=True,
)

# Initiate Logger and log file
logger = logging.getLogger("capstone.gold")

# Log load and initiation
logger.info("Gold Modeling stage starting")

# Log paths loads
log_gold_paths(paths, logger)


In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
# W&B

wandb_run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=WANDB_RUN_NAME,
    job_type="gold_modeling_cascade",
    config={
        "gold_version": GOLD_VERSION,
        "dataset": DATASET_NAME,
        "stage": STAGE,
        "train_fraction": TRAIN_FRACTION,
        "stage1_threshold_percentile": STAGE1_THRESHOLD_PERCENTILE,
        "stage2_threshold_percentile": STAGE2_THRESHOLD_PERCENTILE,
        "gold_input_path": str(GOLD_DATA_PATH),
        "stage1_features_path": str(STAGE1_FEATURES_PATH),
        "stage2_features_path": str(STAGE2_FEATURES_PATH),
        "stage3_primary_path": str(STAGE3_PRIMARY_PATH),
        "stage3_secondary_path": str(STAGE3_SECONDARY_PATH),
    },
)
logger.info("W&B initialized: %s", wandb.run.name)


In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
# Ledger Setup

ledger = Ledger(stage=STAGE, recipe_id=RECIPE_ID)

ledger.add(
    kind="step",
    step="init",
    message="Initialized ledger",
    logger=logger
)


In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
logger.info("Loading Gold parquet: %s", GOLD_DATA_PATH)
gold_working_dataframe = load_data(GOLD_DATA_PATH)

logger.info("Loading Stage 1 features: %s", STAGE1_FEATURES_PATH)
stage1_feature_columns = load_json(STAGE1_FEATURES_PATH)

logger.info("Loading Stage 2 features: %s", STAGE2_FEATURES_PATH)
stage2_feature_columns = load_json(STAGE2_FEATURES_PATH)

logger.info("Loading Stage 3 primary sensors: %s", STAGE3_PRIMARY_PATH)
stage3_primary_rule_sensors = load_json(STAGE3_PRIMARY_PATH)

logger.info("Loading Stage 3 secondary sensors: %s", STAGE3_SECONDARY_PATH)
stage3_secondary_rule_sensors = load_json(STAGE3_SECONDARY_PATH)

ledger.add(
    kind="step",
    step="load_modeling_inputs",
    message="Loaded Gold parquet and stage feature artifacts for modeling.",
    data={
        "gold_path": str(GOLD_DATA_PATH),
        "stage1_features_path": str(STAGE1_FEATURES_PATH),
        "stage2_features_path": str(STAGE2_FEATURES_PATH),
        "stage3_primary_path": str(STAGE3_PRIMARY_PATH),
        "stage3_secondary_path": str(STAGE3_SECONDARY_PATH),
        "gold_shape": list(gold_working_dataframe.shape),
        "stage1_feature_count": int(len(stage1_feature_columns)),
        "stage2_feature_count": int(len(stage2_feature_columns)),
        "stage3_primary_rule_count": int(len(stage3_primary_rule_sensors)),
        "stage3_secondary_rule_count": int(len(stage3_secondary_rule_sensors)),
    },
    logger=logger,
)

gold_working_dataframe.head(3)

In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
def build_time_ordered_split_mask(
    dataframe: pd.DataFrame,
    *,
    train_fraction: float,
) -> tuple[pd.Series, dict]:
    if "time_index" in dataframe.columns:
        ordering_series = dataframe["time_index"].astype("int64")
        ordering_column = "time_index"
    elif "event_step" in dataframe.columns:
        ordering_series = dataframe["event_step"].astype("int64")
        ordering_column = "event_step"
    else:
        ordering_series = pd.Series(np.arange(len(dataframe), dtype=np.int64), index=dataframe.index)
        ordering_column = "row_index"

    sorted_index = ordering_series.sort_values().index
    train_row_count = int(np.floor(len(sorted_index) * train_fraction))

    train_index_values = set(sorted_index[:train_row_count].tolist())
    train_mask = dataframe.index.to_series().apply(lambda index_value: index_value in train_index_values)

    split_info = {
        "train_fraction": float(train_fraction),
        "train_rows": int(train_mask.sum()),
        "test_rows": int((~train_mask).sum()),
        "ordering_column": ordering_column,
    }

    return train_mask, split_info

In [ ]:
train_mask, split_info = build_time_ordered_split_mask(
    gold_working_dataframe,
    train_fraction=TRAIN_FRACTION,
)

ledger.add(
    kind="step",
    step="build_modeling_split",
    message="Rebuilt time-ordered train/test split for Gold modeling notebook.",
    data=split_info,
    logger=logger,
)

split_info

In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
def get_training_rows_for_unsupervised_model(
    dataframe: pd.DataFrame,
    *,
    train_mask: pd.Series,
) -> pd.DataFrame:
    training_subset = dataframe.loc[train_mask].copy()

    if "anomaly_flag" in training_subset.columns:
        training_subset = training_subset[training_subset["anomaly_flag"] == 0].copy()

    return training_subset

In [ ]:
train_rows_for_fit = get_training_rows_for_unsupervised_model(
    gold_working_dataframe,
    train_mask=train_mask,
)

In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
def build_reference_profile(
    dataframe: pd.DataFrame,
    *,
    feature_columns: list[str],
) -> pd.DataFrame:
    reference_rows: list[dict] = []

    for feature_name in feature_columns:
        feature_series = dataframe[feature_name]

        reference_rows.append({
            "feature_name": feature_name,
            "median_value": float(feature_series.median()),
            "mean_value": float(feature_series.mean()),
            "standard_deviation": float(feature_series.std()) if not pd.isna(feature_series.std()) else 0.0,
            "lower_bound": float(feature_series.quantile(0.05)),
            "upper_bound": float(feature_series.quantile(0.95)),
        })

    reference_profile = pd.DataFrame(reference_rows)
    return reference_profile

In [ ]:
reference_profile_features = list(dict.fromkeys(
    stage1_feature_columns + stage3_primary_rule_sensors + stage3_secondary_rule_sensors
))

reference_profile = build_reference_profile(
    train_rows_for_fit,
    feature_columns=reference_profile_features,
)

ledger.add(
    kind="step",
    step="build_reference_profile",
    message="Built training-period reference profile for Stage 3 confirmation logic.",
    data={
        "training_rows": int(len(train_rows_for_fit)),
        "reference_feature_count": int(len(reference_profile_features)),
    },
    logger=logger,
)

reference_profile.head(10)

In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
stage1_train_fit_features = train_rows_for_fit[stage1_feature_columns].values
stage1_all_features = gold_working_dataframe[stage1_feature_columns].values

stage2_train_fit_features = train_rows_for_fit[stage2_feature_columns].values
stage2_all_features = gold_working_dataframe[stage2_feature_columns].values

if "anomaly_flag" in gold_working_dataframe.columns:
    all_labels = gold_working_dataframe["anomaly_flag"].astype(int).values
    test_labels = gold_working_dataframe.loc[~train_mask, "anomaly_flag"].astype(int).values
else:
    all_labels = None
    test_labels = None

In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
def compute_anomaly_scores_isolation_forest(
    model: IsolationForest,
    feature_matrix: np.ndarray,
) -> np.ndarray:
    scores = model.score_samples(feature_matrix)
    anomaly_scores = -scores
    return anomaly_scores

In [ ]:
def choose_threshold_by_percentile(
    anomaly_scores: np.ndarray,
    percentile: float,
) -> float:
    return float(np.percentile(anomaly_scores, percentile))

In [ ]:

def evaluate_against_labels(
    true_labels: np.ndarray,
    anomaly_scores: np.ndarray,
    threshold: float,
) -> dict:
    predicted_labels = (anomaly_scores >= threshold).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        predicted_labels,
        average="binary",
        zero_division=0,
    )

    results = {
        "threshold": float(threshold),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }

    if len(np.unique(true_labels)) == 2:
        results["roc_auc"] = float(roc_auc_score(true_labels, anomaly_scores))
        results["pr_auc"] = float(average_precision_score(true_labels, anomaly_scores))
    else:
        results["roc_auc"] = None
        results["pr_auc"] = None

    return results

In [ ]:
stage1_model = IsolationForest(
    n_estimators=STAGE1_ESTIMATOR_COUNT,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

stage1_model.fit(stage1_train_fit_features)

stage1_train_scores = compute_anomaly_scores_isolation_forest(
    stage1_model,
    train_rows_for_fit[stage1_feature_columns].values,
)

stage1_all_scores = compute_anomaly_scores_isolation_forest(
    stage1_model,
    stage1_all_features,
)

stage1_threshold = choose_threshold_by_percentile(
    stage1_train_scores,
    STAGE1_THRESHOLD_PERCENTILE,
)

stage1_flags = (stage1_all_scores >= stage1_threshold).astype(int)

stage1_summary = {
    "threshold_percentile": STAGE1_THRESHOLD_PERCENTILE,
    "threshold": float(stage1_threshold),
    "alert_count_all_rows": int(stage1_flags.sum()),
    "alert_count_test_rows": int(stage1_flags[~train_mask.values].sum()),
}

#### #### #### #### #### #### #### #### 

ledger.add(
    kind="step",
    step="run_cascade_stage1",
    message="Ran Stage 1 broad Isolation Forest as the cascade screening layer.",
    data={
        "estimator_count": int(STAGE1_ESTIMATOR_COUNT),
        "threshold_percentile": float(STAGE1_THRESHOLD_PERCENTILE),
        "threshold": float(stage1_threshold),
        "feature_count": int(len(stage1_feature_columns)),
        "alert_count_all_rows": int(stage1_summary["alert_count_all_rows"]),
        "alert_count_test_rows": int(stage1_summary["alert_count_test_rows"]),
    },
    logger=logger,
)

#### #### #### #### #### #### #### #### 

stage1_summary

In [ ]:
stage2_model = IsolationForest(
    n_estimators=STAGE2_ESTIMATOR_COUNT,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

stage2_model.fit(stage2_train_fit_features)

stage2_train_scores = compute_anomaly_scores_isolation_forest(
    stage2_model,
    train_rows_for_fit[stage2_feature_columns].values,
)

stage2_all_scores = compute_anomaly_scores_isolation_forest(
    stage2_model,
    stage2_all_features,
)

stage2_threshold = choose_threshold_by_percentile(
    stage2_train_scores,
    STAGE2_THRESHOLD_PERCENTILE,
)

stage2_raw_flags = (stage2_all_scores >= stage2_threshold).astype(int)

# Stage 2 only counts if Stage 1 already flagged
stage2_flags = ((stage1_flags == 1) & (stage2_raw_flags == 1)).astype(int)

stage2_summary = {
    "threshold_percentile": STAGE2_THRESHOLD_PERCENTILE,
    "threshold": float(stage2_threshold),
    "raw_alert_count_all_rows": int(stage2_raw_flags.sum()),
    "stage2_confirmed_count_all_rows": int(stage2_flags.sum()),
    "stage2_confirmed_count_test_rows": int(stage2_flags[~train_mask.values].sum()),
}

#### #### #### #### #### #### #### #### 

ledger.add(
    kind="step",
    step="run_cascade_stage2",
    message="Ran Stage 2 narrow Isolation Forest on the targeted feature subset and gated results through Stage 1.",
    data={
        "estimator_count": int(STAGE2_ESTIMATOR_COUNT),
        "threshold_percentile": float(STAGE2_THRESHOLD_PERCENTILE),
        "threshold": float(stage2_threshold),
        "feature_count": int(len(stage2_feature_columns)),
        "raw_alert_count_all_rows": int(stage2_summary["raw_alert_count_all_rows"]),
        "stage2_confirmed_count_all_rows": int(stage2_summary["stage2_confirmed_count_all_rows"]),
        "stage2_confirmed_count_test_rows": int(stage2_summary["stage2_confirmed_count_test_rows"]),
    },
    logger=logger,
)

#### #### #### #### #### #### #### #### 

stage2_summary

In [ ]:
cascade_results = gold_working_dataframe.copy()

cascade_results["stage1_score"] = stage1_all_scores
cascade_results["stage1_flag"] = stage1_flags

cascade_results["stage2_score"] = stage2_all_scores
cascade_results["stage2_raw_flag"] = stage2_raw_flags
cascade_results["stage2_flag"] = stage2_flags

In [ ]:
def compute_profile_breach_count(
    dataframe: pd.DataFrame,
    *,
    reference_profile: pd.DataFrame,
    feature_columns: list[str],
) -> pd.Series:
    reference_lookup = reference_profile.set_index("feature_name").to_dict("index")
    breach_counts: list[int] = []

    for _, row_values in dataframe.iterrows():
        row_breach_count = 0

        for feature_name in feature_columns:
            if feature_name not in reference_lookup:
                continue

            current_value = row_values[feature_name]
            if pd.isna(current_value):
                continue

            lower_bound = reference_lookup[feature_name]["lower_bound"]
            upper_bound = reference_lookup[feature_name]["upper_bound"]

            if current_value < lower_bound or current_value > upper_bound:
                row_breach_count += 1

        breach_counts.append(row_breach_count)

    return pd.Series(breach_counts, index=dataframe.index, name="stage3_profile_breach_count")

In [ ]:
cascade_results["stage3_profile_breach_count"] = compute_profile_breach_count(
    cascade_results,
    reference_profile=reference_profile,
    feature_columns=stage3_primary_rule_sensors,
)

In [ ]:
def compute_secondary_breach_count(
    dataframe: pd.DataFrame,
    *,
    reference_profile: pd.DataFrame,
    feature_columns: list[str],
) -> pd.Series:
    reference_lookup = reference_profile.set_index("feature_name").to_dict("index")
    breach_counts = pd.Series(0, index=dataframe.index, dtype=int)

    for feature_name in feature_columns:
        if feature_name not in reference_lookup:
            continue

        lower_bound = reference_lookup[feature_name]["lower_bound"]
        upper_bound = reference_lookup[feature_name]["upper_bound"]

        feature_breach_flag = (
            (dataframe[feature_name] < lower_bound) |
            (dataframe[feature_name] > upper_bound)
        ).astype(int)

        breach_counts = breach_counts + feature_breach_flag

    breach_counts.name = "stage3_secondary_breach_count"
    return breach_counts

In [ ]:
cascade_results["stage3_secondary_breach_count"] = compute_secondary_breach_count(
    cascade_results,
    reference_profile=reference_profile,
    feature_columns=stage3_secondary_rule_sensors,
)

In [ ]:
def compute_persistence_flag(
    source_flags: pd.Series,
    *,
    rolling_window_size: int = 3,
    minimum_flags_in_window: int = 2,
) -> pd.Series:
    persistence_flag = (
        source_flags
        .rolling(window=rolling_window_size, min_periods=1)
        .sum()
        .ge(minimum_flags_in_window)
        .astype(int)
    )

    persistence_flag.name = "stage3_persistence_flag"
    return persistence_flag

In [ ]:
cascade_results["stage3_persistence_flag"] = compute_persistence_flag(
    cascade_results["stage2_flag"],
    rolling_window_size=3,
    minimum_flags_in_window=2,
)


In [ ]:
def compute_drift_flag(
    dataframe: pd.DataFrame,
    *,
    feature_columns: list[str],
    rolling_window_size: int = 5,
    drift_threshold_multiplier: float = 1.0,
) -> pd.Series:
    drift_trigger_counts = pd.Series(0, index=dataframe.index, dtype=int)

    for feature_name in feature_columns:
        feature_series = dataframe[feature_name]
        feature_standard_deviation = feature_series.std()

        if pd.isna(feature_standard_deviation) or feature_standard_deviation == 0:
            continue

        rolling_median = feature_series.rolling(window=rolling_window_size, min_periods=1).median()
        rolling_delta = (feature_series - rolling_median).abs()

        feature_drift_flag = (
            rolling_delta > (feature_standard_deviation * drift_threshold_multiplier)
        ).astype(int)

        drift_trigger_counts = drift_trigger_counts + feature_drift_flag

    drift_flag = (drift_trigger_counts >= 1).astype(int)
    drift_flag.name = "stage3_drift_flag"
    return drift_flag

In [ ]:
stage3_rule_watch_features = list(dict.fromkeys(
    stage3_primary_rule_sensors + stage3_secondary_rule_sensors
))

cascade_results["stage3_drift_flag"] = compute_drift_flag(
    cascade_results,
    feature_columns=stage3_rule_watch_features,
    rolling_window_size=5,
    drift_threshold_multiplier=1.0,
)

In [ ]:

cascade_results["stage3_profile_breach_flag"] = (
    cascade_results["stage3_profile_breach_count"] >= 2
).astype(int)

cascade_results["stage3_corroboration_flag"] = (
    cascade_results["stage3_secondary_breach_count"] >= 1
).astype(int)

cascade_results["stage3_rule_evidence_count"] = (
    cascade_results["stage3_profile_breach_flag"] +
    cascade_results["stage3_persistence_flag"] +
    cascade_results["stage3_drift_flag"] +
    cascade_results["stage3_corroboration_flag"]
)

cascade_results["cascade_final_flag"] = (
    (cascade_results["stage1_flag"] == 1) &
    (cascade_results["stage2_flag"] == 1) &
    (
        (cascade_results["stage3_profile_breach_count"] >= 2) |
        (cascade_results["stage3_rule_evidence_count"] >= 2)
    )
).astype(int)


stage3_summary = {
    "primary_rule_sensor_count": int(len(stage3_primary_rule_sensors)),
    "secondary_rule_sensor_count": int(len(stage3_secondary_rule_sensors)),
    "profile_breach_rows": int((cascade_results["stage3_profile_breach_flag"] == 1).sum()),
    "corroboration_rows": int((cascade_results["stage3_corroboration_flag"] == 1).sum()),
    "persistence_rows": int((cascade_results["stage3_persistence_flag"] == 1).sum()),
    "drift_rows": int((cascade_results["stage3_drift_flag"] == 1).sum()),
    "final_alert_count_all_rows": int(cascade_results["cascade_final_flag"].sum()),
    "final_alert_count_test_rows": int(cascade_results.loc[~train_mask, "cascade_final_flag"].sum()),
}

ledger.add(
    kind="step",
    step="run_cascade_stage3_confirmation",
    message="Applied Stage 3 rule, profile, persistence, drift, and corroboration checks to confirm final cascade alerts.",
    data=stage3_summary,
    logger=logger,
)

cascade_results.head(5)

In [ ]:
cascade_metrics = {
    "model": "3-Stage Cascade",
    "stage1_alert_count_all_rows": int(cascade_results["stage1_flag"].sum()),
    "stage2_alert_count_all_rows": int(cascade_results["stage2_flag"].sum()),
    "final_alert_count_all_rows": int(cascade_results["cascade_final_flag"].sum()),
    "stage1_alert_count_test_rows": int(cascade_results.loc[~train_mask, "stage1_flag"].sum()),
    "stage2_alert_count_test_rows": int(cascade_results.loc[~train_mask, "stage2_flag"].sum()),
    "final_alert_count_test_rows": int(cascade_results.loc[~train_mask, "cascade_final_flag"].sum()),
}

if test_labels is not None:
    cascade_test_flags = cascade_results.loc[~train_mask, "cascade_final_flag"].astype(int).values

    precision, recall, f1, _ = precision_recall_fscore_support(
        test_labels,
        cascade_test_flags,
        average="binary",
        zero_division=0,
    )

    cascade_metrics["precision"] = float(precision)
    cascade_metrics["recall"] = float(recall)
    cascade_metrics["f1"] = float(f1)

cascade_metrics

In [ ]:
stage1_alert_count_all_rows = int(cascade_results["stage1_flag"].sum())
stage2_alert_count_all_rows = int(cascade_results["stage2_flag"].sum())
final_cascade_alert_count_all_rows = int(cascade_results["cascade_final_flag"].sum())

stage1_alert_count_test_rows = int(cascade_results.loc[~train_mask, "stage1_flag"].sum())
stage2_alert_count_test_rows = int(cascade_results.loc[~train_mask, "stage2_flag"].sum())
final_cascade_alert_count_test_rows = int(cascade_results.loc[~train_mask, "cascade_final_flag"].sum())

cascade_thresholds = {
    "stage1_threshold_percentile": float(STAGE1_THRESHOLD_PERCENTILE),
    "stage1_threshold": float(stage1_threshold),
    "stage2_threshold_percentile": float(STAGE2_THRESHOLD_PERCENTILE),
    "stage2_threshold": float(stage2_threshold),
}


cascade_summary = {
    "dataset_name": DATASET_NAME,
    "cascade_metrics": cascade_metrics,
    "stage1_alert_count_all_rows": stage1_alert_count_all_rows,
    "stage2_alert_count_all_rows": stage2_alert_count_all_rows,
    "final_cascade_alert_count_all_rows": final_cascade_alert_count_all_rows,
    "stage1_alert_count_test_rows": stage1_alert_count_test_rows,
    "stage2_alert_count_test_rows": stage2_alert_count_test_rows,
    "final_cascade_alert_count_test_rows": final_cascade_alert_count_test_rows,
    "result_row_count": int(len(cascade_results)),
    "stage1_feature_count": int(len(stage1_feature_columns)),
    "stage2_feature_count": int(len(stage2_feature_columns)),
    "stage3_primary_rule_count": int(len(stage3_primary_rule_sensors)),
    "stage3_secondary_rule_count": int(len(stage3_secondary_rule_sensors)),
}

cascade_metadata = {
    "cascade_results_path": str(CASCADE_RESULTS_PATH),
    "stage1_model_path": str(STAGE1_MODEL_PATH),
    "stage2_model_path": str(STAGE2_MODEL_PATH),
    "cascade_thresholds_path": str(CASCADE_THRESHOLDS_PATH),
    "cascade_summary_path": str(CASCADE_SUMMARY_PATH),
    "cascade_reference_profile_path": str(CASCADE_REFERENCE_PROFILE_PATH),
}

cascade_results.to_csv(CASCADE_RESULTS_PATH, index=False)
reference_profile.to_csv(CASCADE_REFERENCE_PROFILE_PATH, index=False)

joblib.dump(stage1_model, STAGE1_MODEL_PATH)
joblib.dump(stage2_model, STAGE2_MODEL_PATH)

save_json(cascade_thresholds, CASCADE_THRESHOLDS_PATH)
save_json(cascade_summary, CASCADE_SUMMARY_PATH)
save_json(cascade_metadata, CASCADE_METADATA_PATH)

wandb.save(str(CASCADE_RESULTS_PATH))
wandb.save(str(CASCADE_REFERENCE_PROFILE_PATH))
wandb.save(str(STAGE1_MODEL_PATH))
wandb.save(str(STAGE2_MODEL_PATH))
wandb.save(str(CASCADE_THRESHOLDS_PATH))
wandb.save(str(CASCADE_SUMMARY_PATH))
wandb.save(str(CASCADE_METADATA_PATH))

ledger.add(
    kind="step",
    step="save_cascade_outputs",
    message="Saved cascade results, trained Stage 1 and Stage 2 models, thresholds, summary, metadata, and reference profile.",
    data={
        "cascade_results_path": str(CASCADE_RESULTS_PATH),
        "cascade_reference_profile_path": str(CASCADE_REFERENCE_PROFILE_PATH),
        "stage1_model_path": str(STAGE1_MODEL_PATH),
        "stage2_model_path": str(STAGE2_MODEL_PATH),
        "cascade_thresholds_path": str(CASCADE_THRESHOLDS_PATH),
        "cascade_summary_path": str(CASCADE_SUMMARY_PATH),
        "cascade_metadata_path": str(CASCADE_METADATA_PATH),
        "result_row_count": int(len(cascade_results)),
        "stage1_alert_count_all_rows": stage1_alert_count_all_rows,
        "stage2_alert_count_all_rows": stage2_alert_count_all_rows,
        "final_cascade_alert_count_all_rows": final_cascade_alert_count_all_rows,
    },
    logger=logger,
)

In [ ]:
ledger.add(
    kind="step",
    step="finalize_cascade_modeling",
    message="Gold cascade modeling notebook complete.",
    data={
        "cascade_metrics": cascade_metrics,
        "cascade_results_path": str(CASCADE_RESULTS_PATH),
        "stage1_model_path": str(STAGE1_MODEL_PATH),
        "stage2_model_path": str(STAGE2_MODEL_PATH),
    },
    logger=logger,
)

cascade_ledger_path = GOLD_ARTIFACTS_PATH / f"ledger__{DATASET_NAME}__gold_cascade_modeling.json"
ledger.write_json(cascade_ledger_path)

wandb.save(str(cascade_ledger_path))
wandb_run.finish()